# 编队环境下的PE计算前置

改进点：

1. 使用分段线性位移变换方案改良了PE（考虑是否需要和基准PE进行对比试验）
2. 通过线性滤波改善大角度情况下产生的步进误差。

完成基准测试：

1. 双射线模型与PE在平坦海面的降级测试
2. fdtd与pe在单一山丘环境下的降级测试
3. fdtd与pe在[1,1.5,2,2.5,3,3.5,4,4.5,5]环境下的全波对比。

以上均方根误差RMSE均在0.8-2.2dB之间，效果较好


# 本实验目的

通过前置实验已经验证了基准测试下PE方程计算的有效性，PE求解器和FDTD求解器的正确性。

本部分实验进而将PE推导应用到编队场景。得到四个指标的计算结果，注意需要时评价体系完备且正向。


# 四个指标


1.	系统级电磁耦合度 (System Coupling Factor, SCF)

传统的干扰裕度仅评估单条链路的通断，难以反映编队整体的电磁拥堵状况。SCF从全局能量维视角出发，量化整个编队系统内部各节点间的非预期能量耦合总水平。该指标越低，表明编队内的频谱规划越合理，系统自身的电磁兼容性越好。

首先基于向量模型，遍历编队内所有$N$个发射源与$M$个接收机节点。利用传播模型计算每一对节点间的接收功率$P_{rx}^{i,j}$。构建系统耦合矩阵${{\mathbf{C}}_{M\times N}}$，其中元素 代表第j个发射源对第i个非目标接收机的干扰功率。计算该矩阵的平均能量水平作为SCF值。

$$	SCF=\frac{1}{M\times N}\sum\limits_{i=1}^{M}{\sum\limits_{j=1}^{N}{\left( P_{rx}^{i,j}-{{N}_{floor}} \right)}}$$

式中，$P_{rx}^{i,j}$ 为接收功率（dBm），${{N}_{floor}}$为系统底噪，$j\ne i$（排除自身通信链路）。

2.	海况敏感度指数 (Sea-State Sensitivity Index, S3I)

鉴于海面多径效应是海上电磁传播的主要特征，本指标用于定量评估海面粗糙度（由风浪等级决定）对编队电磁环境稳定性的影响。S3I指数越高，说明当前编队队形和频段配置受海况变化影响越剧烈，需要考虑恶劣海况下的余量设计。这也验证了引入 JONSWAP 谱与 PLST 算法的必要性。

设定 JONSWAP 谱参数为“平静海面”（波高$\approx 0$），计算基准场分布功率${{P}_{flat}}$。

设定 JONSWAP 谱参数为“指定海况”（如5级海况），应用 PLST 修正计算粗糙海面场分布功率${{P}_{rough}}$。

最后统计编队活动区域内所有采样点$k$的功率差异绝对值。并作统计对照处理

$$S3I=\frac{1}{K}\sum\limits_{k=1}^{K}{\left| {{P}_{rough}}({{x}_{k}},{{y}_{k}})-{{P}_{flat}}({{x}_{k}},{{y}_{k}}) \right|}$$

式中，$K$为空间采样点总数。

3.	背景噪声抬升热图 (Background Noise Elevation Map)

该指标借鉴认知无线电中的“干扰温度”概念，从频谱管理的角度评估环境的电磁强度。它不依赖于具体的接收机位置，而是生成一张覆盖整个海域的热力图，直观展示由于编队所有设备发射导致的电磁背景噪声相对于自然热噪声的抬升倍数。该热图可用于识别编队内部电磁热点区域。

首先，在海域平面$(x,y)$上，通过非相干叠加计算所有发射源在该点产生的总干扰功率${{I}_{total}}(x,y)$。根据玻尔兹曼常数与系统带宽，确定自然环境下的热噪声基底${{N}_{thermal}}$。计算各点总干扰功率与热噪声的比值，并映射为色阶。

$$	{{T}_{elevation}}(x,y)=10{{\log }_{10}}\left( \frac{{{I}_{total}}(x,y)+{{N}_{thermal}}}{{{N}_{thermal}}} \right) $$

4.	接收机灵敏度恶化分布图 (Receiver Desensitization Map)

这是面向作战效能的工程指标。当环境干扰电平超过接收机固有的灵敏度阈值时，接收机的实际探测能力将被迫下降（即“灵敏度恶化”）。该分布图直观地回答了指挥员雷达或通信设备可能会失效的问题，直接关联战术部署安全。

读取受害设备模型参数中的接收机灵敏度阈值${{S}_{sens}}$（例如 -90dBm）。对比该点的总干扰场强${{I}_{total}}(x,y)$与灵敏度阈值。计算超出阈值的部分，即为灵敏度恶化量（Desense）。

$$D_{\text{desense}}(x,y) = \begin{cases} 
I_{\text{total}}(x,y) - S_{\text{sens}}, & \text{if } I_{\text{total}}(x,y) > S_{\text{sens}} \\
0, & \text{otherwise}
\end{cases}$$

本设计最后目标提出一套适用于无人船编队的电磁兼容计算方法，涵盖工况遴选、干扰等效、传播建模、数据融合的完整流程。相比既有单平台EMC分析方法，本方案显著增强了对多平台协同干扰的处理能力。

利用仿真结果提出编队系统的EMC优化建议，例如调整编队队形增大安全距离、采用频谱分离减少同频干扰、加强易扰设备的屏蔽隔离等。根据某型无人艇编队的仿真分析，验证软件在指导设计改进方面的价值。



In [37]:
# 导入包
import meep as mp
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"] + plt.rcParams["font.serif"],
    "mathtext.fontset": "stix",
})
from mpl_toolkits.axes_grid1 import ImageGrid
import scipy.fft as fft
import pandas as pd
from abc import ABC, abstractmethod
from scipy.ndimage import uniform_filter1d 

# ==========================================
# 场景生成器 (Scene Generator)
# 定义全局物理参数、生成海面、统一下发坐标
# ==========================================
class SceneGenerator:
    def __init__(self, 
                 freq_ghz=0.3,      # 频率(GHz)
                 lx=320.0,          # 仿真域长度 (m)
                 lz=50.0,           # 仿真域高度 (m)
                 dpml=5.0,          # 吸收层厚度 (m)
                 wind_speed=15.0,   # 风速 (m/s)
                 fetch_km=50.0,     # 风区 (km)
                 tx_height=5.0,     # 发射机高度 (相对海平面, m)
                 rx_height=5.0):    # 接收机观测高度 (相对海平面, m)
        self.freq_ghz = freq_ghz
        self.lx = lx        
        self.lz = lz
        self.dpml = dpml
        self.wind_speed = wind_speed
        self.fetch_km = fetch_km
        self.tx_height = tx_height
        self.rx_height = rx_height
        self.base_water_level = 0.0# 绝对坐标系下的平均海平面位置
        
        # FDTD 网格分辨率计算
        self.resolution = 10        # FDTD 网格分辨率
        self.dx_fdtd = 1.0 / self.resolution
        
        # 统一生成基于 JONSWAP 的全域海面
        self.x_full, self.h_full = self._generate_jonswap()

    def _generate_jonswap(self):
        g = 9.81              
        fetch_m = self.fetch_km * 1000.0
        X_tilde = (g * fetch_m) / (self.wind_speed**2)
        wp = 22 * (g / self.wind_speed) * (X_tilde**(-0.33))
        alpha = 0.076 * (X_tilde**(-0.22))
        
        length_m = self.lx + 2 * self.dpml
        k_min = 2 * np.pi / length_m
        k_max = 2 * np.pi / (2 * self.dx_fdtd)
        dk = 2 * np.pi / length_m
        k_arr = np.arange(k_min, k_max, dk)
        w_arr = np.sqrt(g * k_arr)
        
        S_pm = (alpha * g**2 / (w_arr**5)) * np.exp(-1.25 * (wp / w_arr)**4)
        gamma = 3.3  
        sigma = np.where(w_arr <= wp, 0.07, 0.09)
        r = np.exp(-(w_arr - wp)**2 / (2 * sigma**2 * wp**2))
        enhancement = gamma ** r
        S_jonswap = S_pm * enhancement
        
        dw = np.diff(w_arr, prepend=w_arr[0])
        amplitudes = np.sqrt(2 * S_jonswap * dw)
        
        np.random.seed(42) # 固定随机种子重复实验
        phases = np.random.uniform(0, 2*np.pi, size=len(k_arr))
        
        x = np.arange(0, length_m, self.dx_fdtd)
        h = np.zeros_like(x)
        
        print(f"✅ 场景生成完毕: JONSWAP海面 (风速={self.wind_speed}m/s)")
        for i in range(len(k_arr)):
            h += amplitudes[i] * np.cos(k_arr[i] * x + phases[i])
            
        return x, h

In [38]:
# ==========================================
# FDTD 求解器接口 (FDTD Solver)
# ==========================================
class FDTDSolver:
    def __init__(self, scene: SceneGenerator):
        self.scene = scene

    def run(self):
        mp.verbosity(0)
        
        # ── 坐标系说明 ──────────────────────────────────────────────
        # 绝对坐标 (abs): x_full 的原始坐标，范围 [0, lx+2*dpml]
        # Meep 坐标 (meep): 以仿真域中心为原点，= abs - x_center
        # 物理坐标 (phys): 相对于左侧 PML 边界，= abs - dpml
        # ────────────────────────────────────────────────────────────
        x_center = np.mean(self.scene.x_full)   # ≈ (lx + 2*dpml) / 2
        x_meep   = self.scene.x_full - x_center

        # ── 地形几何体 ──────────────────────────────────────────────
        sea_geometry = []
        floor_z = -self.scene.lz / 2 - self.scene.dpml
        for i in range(len(x_meep) - 1):
            z_val1 = min(self.scene.base_water_level + self.scene.h_full[i],
                         self.scene.lz / 2 - self.scene.dpml - 0.5)
            z_val2 = min(self.scene.base_water_level + self.scene.h_full[i + 1],
                         self.scene.lz / 2 - self.scene.dpml - 0.5)
            v1 = mp.Vector3(x_meep[i],     floor_z)
            v2 = mp.Vector3(x_meep[i + 1], floor_z)
            v3 = mp.Vector3(x_meep[i + 1], z_val2)
            v4 = mp.Vector3(x_meep[i],     z_val1)
            sea_geometry.append(mp.Prism([v1, v2, v3, v4],
                                         height=mp.inf, material=mp.metal))

        cell_size      = mp.Vector3(self.scene.lx + 2 * self.scene.dpml,
                                    self.scene.lz + 2 * self.scene.dpml)
        boundary_layers = [mp.PML(self.scene.dpml)]

        # ── 频率转换 ────────────────────────────────────────────────
        c_light      = 299792458.0
        wavelength_m = c_light / (self.scene.freq_ghz * 1e9)
        freq_meep    = 1.0 / wavelength_m

        # ── 源位置（坐标对齐修复保留）───────────────────────────────
        tx_physical_x = 10.0
        tx_abs_x      = self.scene.dpml + tx_physical_x   # 绝对坐标 = 15.0
        tx_meep_x     = tx_abs_x - x_center               # Meep 坐标
        tx_z_meep     = self.scene.base_water_level + self.scene.tx_height

        # ✅ 恢复：各向同性点源（正确的柱面波物理模型）
        sources = [mp.Source(
            mp.ContinuousSource(frequency=freq_meep),
            component=mp.Ez,
            center=mp.Vector3(tx_meep_x, tx_z_meep),
            size=mp.Vector3(0, 0)    # ✅ 点源，不是线源
        )]

        sim = mp.Simulation(
            cell_size=cell_size,
            boundary_layers=boundary_layers,
            geometry=sea_geometry,
            sources=sources,
            resolution=self.scene.resolution,
            force_complex_fields=True
        )

        steady_state_time = self.scene.lx * 5
        print(f"⏳ 开始 FDTD 仿真 (预计达到稳态时间: {steady_state_time})...")
        sim.run(until=steady_state_time)

        # ── 提取场数据 ──────────────────────────────────────────────
        ez_data = sim.get_array(center=mp.Vector3(), size=cell_size, component=mp.Ez)

        # 坐标轴：从 0 到 cell_size.x（绝对坐标）
        x_coords_full = np.linspace(0, cell_size.x, ez_data.shape[0])
        z_coords_full = np.linspace(-cell_size.y / 2, cell_size.y / 2, ez_data.shape[1])

        # ✅ 修复：提取起点 = 源的绝对坐标 tx_abs_x（而非 dpml+tx_physical_x 的旧错误）
        abs_end_x = self.scene.dpml + self.scene.lx
        tx_x_idx  = np.argmin(np.abs(x_coords_full - tx_abs_x))   # ✅ 与源位置对齐
        end_idx   = np.argmin(np.abs(x_coords_full - abs_end_x))
        z_idx     = np.argmin(np.abs(z_coords_full - tx_z_meep))   # 接收高度

        fdtd_range    = x_coords_full[tx_x_idx:end_idx] - x_coords_full[tx_x_idx]
        fdtd_1d_mag   = np.abs(ez_data[tx_x_idx:end_idx, z_idx])
        fdtd_2d_mag   = np.abs(ez_data[tx_x_idx:end_idx, :])
        z_physical_coords = z_coords_full - self.scene.base_water_level

        print(f"✅ FDTD 全波解计算完毕，有效长度: {fdtd_range[-1]:.2f}m")
        
        # ✅ 修复：返回 tx_abs_x，使 PE 侧能精确对齐起点
        return fdtd_range, fdtd_1d_mag, fdtd_2d_mag, z_physical_coords, tx_abs_x

In [39]:
import numpy as np
import scipy.fft as fft
from scipy.ndimage import uniform_filter1d

# ==========================================
# 优化后极度稳定的 PE 求解器 (PESolver)
# ==========================================
class PESolver:
    def __init__(self, scene, dx=0.1, dz=0.1):
        self.c = 299792458.0
        self.freq = scene.freq_ghz * 1e9
        self.k0 = 2 * np.pi * self.freq / self.c
        self.dx = dx
        self.dz = dz
        self.max_z = scene.lz
        self.computation_lz = scene.lz * 1.5 
        self.nz = int(self.computation_lz / dz)
        self.fft_size = 2 * self.nz 
        self.z = np.arange(self.nz) * self.dz
        self.kz = fft.fftfreq(self.fft_size, d=self.dz) * 2 * np.pi
        self.u = np.zeros(self.fft_size, dtype=np.complex128)
        self._setup_absorber()

    def _setup_absorber(self):
        self.absorber = np.ones(self.nz)
        absorb_layer_thickness = int(self.nz * 0.25)
        start_idx = self.nz - absorb_layer_thickness
        window = 0.5 * (1 + np.cos(np.pi * np.arange(absorb_layer_thickness) / absorb_layer_thickness))
        self.absorber[start_idx:] = window

    def init_gaussian_source(self, antenna_z_phys, h_surf_0, beam_width=0.2):
        zeta_a = antenna_z_phys - h_surf_0
        self.u[:self.nz] = np.exp(-((self.z - zeta_a)**2) / (2 * beam_width**2))
        
        # ✅ 强制奇对称（下边界完美反射条件）
        self.u[self.nz + 1:] = -self.u[self.nz - 1: 0: -1]
        self.u[0] = 0.0
        self.u[self.nz] = 0.0
        
        # k 域低通滤波，滤除无法传播的超大角度能量
        kz_filter = np.exp(-(self.kz / (0.9 * self.k0))**10)
        self.u = fft.ifft(fft.fft(self.u) * kz_filter)
        
        # 滤波后再次强制奇对称，确保万无一失
        self.u[self.nz + 1:] = -self.u[self.nz - 1: 0: -1]
        self.u[0] = 0.0
        self.u[self.nz] = 0.0

    def march(self, x_surf, h_surf, max_range, receiver_z_phys, smooth_window=10):
        # print(f"⏳ 开始 PE 传播步进... (Δz = {self.dz}m)")
        h_surf_smoothed = uniform_filter1d(h_surf, size=smooth_window, mode='nearest')
        
        physical_nz = int(self.max_z / self.dz)
        results_x    = [0.0]
        results_2d   = [np.abs(self.u[:physical_nz])] # ✅ 修复：初始化时即剔除吸收层
        h_surf_pe    = [h_surf_smoothed[0]]

        idx_rx_0 = int((receiver_z_phys - h_surf_smoothed[0]) / self.dz)
        E0 = np.abs(self.u[idx_rx_0]) if 0 <= idx_rx_0 < self.nz else 1e-12
        results_E_mag = [E0]

        steps = int(max_range / self.dx)
        for s in range(1, steps + 1):
            x_curr = (s - 1) * self.dx
            x_next = s * self.dx

            z_curr = np.interp(x_curr, x_surf, h_surf_smoothed)
            z_next = np.interp(x_next, x_surf, h_surf_smoothed)
            slope  = (z_next - z_curr) / self.dx
            beta   = np.arctan(slope)

            # --- 折射与边界条件 ---
            gamma = -1.0 + 0j
            val_ref    = np.cos(beta)**2 + 0j
            refraction = np.exp(1j * self.k0 * self.dx * (np.sqrt(val_ref) - 1.0))

            self.u[:self.nz] = self.u[:self.nz] * refraction * self.absorber
            self.u[self.nz + 1:] = gamma * self.u[self.nz - 1: 0: -1]
            self.u[0]     *= (1.0 + gamma)
            self.u[self.nz] = 0.0

            # --- 衍射传播 (SSFT) ---
            k_eff_sq  = (self.k0 * np.cos(beta))**2
            val_diff  = k_eff_sq - self.kz**2 + 0j
            
            # ✅ 修复：解决浮点误差导致倏逝波指数爆炸的“条状阴影”问题
            sqrt_val = np.sqrt(val_diff)
            sqrt_val = np.real(sqrt_val) + 1j * np.abs(np.imag(sqrt_val))
            diffraction = np.exp(1j * self.dx * (sqrt_val - self.k0 * np.cos(beta)))

            u_k = fft.fft(self.u)
            window_k = np.exp(-(self.kz / (0.95 * self.k0))**10)
            u_k = u_k * diffraction * window_k
            self.u = fft.ifft(u_k)

            # --- 记录数据 ---
            E_mag_2d = np.abs(self.u[:physical_nz])
            results_x.append(x_next)
            results_2d.append(E_mag_2d)
            h_surf_pe.append(z_next)

            zeta_rx = receiver_z_phys - z_next
            idx = int(zeta_rx / self.dz) if 0 <= zeta_rx < self.max_z else -1
            results_E_mag.append(E_mag_2d[idx] if idx != -1 else 1e-12)

        x_arr = np.array(results_x)
        E_arr = np.array(results_E_mag)

        return (x_arr, E_arr, np.array(results_2d).T, self.z[:physical_nz], np.array(h_surf_pe))

In [40]:

# ==========================================
# 评估器与数学计算工具 (Metrics Evaluator)
# ==========================================
class MetricsEvaluator:
    @staticmethod
    def align_and_convert_to_dB(pe_mag, fdtd_mag, pe_range, fdtd_range):
        pe_dB = 20 * np.log10(pe_mag + 1e-12)
        fdtd_dB = 20 * np.log10(fdtd_mag + 1e-12)

        # 全局平移对齐 (基于 50m~150m 远场计算系统偏置误差)
        align_idx_pe = np.where((pe_range > 50) & (pe_range < 150))[0]
        align_idx_fdtd = np.where((fdtd_range > 50) & (fdtd_range < 150))[0]
        offset_dB = np.mean(pe_dB[align_idx_pe]) - np.mean(fdtd_dB[align_idx_fdtd])
        
        return pe_dB, fdtd_dB + offset_dB, offset_dB

    @staticmethod
    def calc_rmse_with_protection(pe_dB, fdtd_dB_aligned, pe_range, fdtd_range,
                                   min_range=50.0, threshold=-65.0):
        """零点保护 RMSE：跳过近场和深零点"""
        pe_dB_interp = np.interp(fdtd_range, pe_range, pe_dB)
        mask = (fdtd_range > min_range) & (fdtd_dB_aligned > threshold)
        if not np.any(mask):
            return 0.0
        return np.sqrt(np.mean((pe_dB_interp[mask] - fdtd_dB_aligned[mask])**2))


    @staticmethod
    def calc_cumulative_rmse(pe_dB, fdtd_dB_aligned, pe_range, fdtd_range, min_range=20.0):
        pe_dB_interp = np.interp(fdtd_range, pe_range, pe_dB)
        cum_rmse = np.full_like(fdtd_range, np.nan)
        for i in range(len(fdtd_range)):
            if fdtd_range[i] > min_range:
                valid_idx = np.where((fdtd_range > min_range) & (fdtd_range <= fdtd_range[i]))[0]
                if len(valid_idx) > 0:
                    cum_rmse[i] = np.sqrt(np.mean((pe_dB_interp[valid_idx] - fdtd_dB_aligned[valid_idx])**2))
        return cum_rmse


In [41]:

# ==========================================
# 4. 可视化组件层 (Visualizers conforming to OCP)
# ==========================================
class BaseVisualizer(ABC):
    @abstractmethod
    def plot(self, *args, **kwargs):
        pass

class HeatmapVisualizer(BaseVisualizer):
    def plot(self, fdtd_2d_mag, pe_2d_mapped, fdtd_range, pe_range, z_coords, offset_dB):
        fig  = plt.figure(figsize=(10, 8))
        grid = ImageGrid(fig, 111, nrows_ncols=(2, 1), axes_pad=0.4,
                         share_all=True, cbar_location="right",
                         cbar_mode="single", cbar_pad=0.1)

        fdtd_2d_dB = 20 * np.log10(fdtd_2d_mag + 1e-12) + offset_dB
        pe_2d_dB   = 20 * np.log10(pe_2d_mapped  + 1e-12)
        vmin, vmax = -80, -20

        extent = [fdtd_range[0], fdtd_range[-1], z_coords[0], z_coords[-1]]
        im1 = grid[0].imshow(fdtd_2d_dB.T, extent=extent, origin='lower',
                              aspect='auto', cmap='jet', vmin=vmin, vmax=vmax)
        grid[0].set_title('FDTD 2D Field (Aligned)')
        grid[0].set_ylabel('Height (m)')

        im2 = grid[1].imshow(pe_2d_dB, extent=extent, origin='lower',
                              aspect='auto', cmap='jet', vmin=vmin, vmax=vmax)
        grid[1].set_title('PE (PLST) 2D Field')
        grid[1].set_xlabel('Range (m)')
        grid[1].set_ylabel('Height (m)')

        grid[0].cax.colorbar(im1)
        plt.savefig('Fig1_Heatmap_Comparison.png', dpi=300, bbox_inches='tight')
        plt.close()

class MultiWindSpeedVisualizer(BaseVisualizer):
    def plot(self, results_dict):
        """ results_dict: {wind_speed: (fdtd_range, fdtd_dB, pe_range, pe_dB, rmse)} """
        plt.figure(figsize=(12, 8))
        colors = {1: 'g',1.5: 'g',2: 'g',2.5: 'g',3: 'g',3.5: 'g',4: 'g',4.5: 'g',5: 'g'}
        
        for ws, data in results_dict.items():
            fdtd_range, fdtd_dB, pe_range, pe_dB, rmse = data
            c = colors[ws]
            plt.plot(fdtd_range, fdtd_dB, c=c, linestyle='-', alpha=0.5, label=f'FDTD (WS={ws}m/s)')
            plt.plot(pe_range, pe_dB, c=c, linestyle='--', label=f'PE (WS={ws}m/s, RMSE={rmse:.2f}dB)')
            
        plt.title('Normalized Field Strength Comparison at Different Wind Speeds')
        plt.xlabel('Range (m)')
        plt.ylabel('Normalized Field Strength (dB)')
        plt.ylim([-90, -20])
        plt.legend(loc='lower left')
        plt.tight_layout()
        plt.savefig('Fig2_WindSpeed_Comparison.png', dpi=300)
        plt.close()

class CumulativeRMSEVisualizer(BaseVisualizer):
    def plot(self, results_dict):
        plt.figure(figsize=(10, 6))
        colors = {1: 'g',1.5: 'g',2: 'g',2.5: 'g',3: 'g',3.5: 'g',4: 'g',4.5: 'g',5: 'g'}
        
        for ws, data in results_dict.items():
            fdtd_range, fdtd_dB, pe_range, pe_dB, _ = data
            cum_rmse = MetricsEvaluator.calc_cumulative_rmse(pe_dB, fdtd_dB, pe_range, fdtd_range)
            plt.plot(fdtd_range, cum_rmse, c=colors[ws], linewidth=2, label=f'Cum. RMSE (WS={ws}m/s)')
            
        plt.title('Cumulative RMSE vs. Range')
        plt.xlabel('Range (m)')
        plt.ylabel('Cumulative RMSE (dB)')
        plt.grid(True, linestyle=':', alpha=0.7)
        plt.legend()
        plt.tight_layout()
        plt.savefig('Fig3_Cumulative_RMSE.png', dpi=300)
        plt.close()


In [42]:
# ==========================================
# 6. 编队场景生成器 (Formation Scene Generator)
#    解析 Test.json，构建编队节点模型
# ==========================================
import json as _json, re as _re

class FormationSceneGenerator:
    def __init__(self, json_path, wind_speed=3.0, lz=50.0, dpml=5.0, fetch_km=50.0):
        self.wind_speed = wind_speed
        self.lz = lz
        self.dpml = dpml
        self.fetch_km = fetch_km
        # 读取 JSON：去行注释，处理 UTF-8 BOM
        with open(json_path, 'r', encoding='utf-8-sig') as f:
            raw = f.read()
        clean = _re.sub(r'//[^\n]*', '', raw)
        self.config = _json.loads(clean)
        self.nodes = self._parse_nodes()
        # 公共频率/带宽从第一个发射机获取
        first_usv = next(iter(self.config.values()))
        tx0 = next(v for v in first_usv.values()
                   if isinstance(v, dict) and v.get('type') == 'TRANSMITTER')
        self.freq_ghz = float(tx0['Central_F'])
        self.bw_mhz   = float(tx0['Bandwith'])
        print(f"✅ 编队场景加载完毕: {len(self.nodes)} 个 USV | "
              f"f={self.freq_ghz} GHz | BW={self.bw_mhz} MHz | WS={wind_speed} m/s")

    def _parse_nodes(self):
        """
        解析每个 USV 下的发射机和接收机，
        绝对坐标 = Location.coordinates + Location_Offset
        """
        nodes = {}
        for usv_id, usv_data in self.config.items():
            loc = usv_data['Location']['coordinates']   # [x, y, z] m
            txs, rxs = {}, {}
            for key, dev in usv_data.items():
                if not isinstance(dev, dict):
                    continue
                dev_type = dev.get('type', '')
                if dev_type not in ('TRANSMITTER', 'RECEIVER'):
                    continue
                off = dev.get('Location_Offset', [0, 0, 0])
                pos = [float(loc[i]) + float(off[i]) for i in range(3)]

                if dev_type == 'TRANSMITTER':
                    txs[key] = {
                        'id'            : dev['ID'],
                        'position'      : pos,                         # [x,y,z] m
                        'power_dbm'     : float(dev['Power']),         # dBm
                        'gain_dbi'      : float(dev['Gain']),          # dBi
                        'freq_ghz'      : float(dev['Central_F']),     # GHz
                        'bw_mhz'        : float(dev['Bandwith']),      # MHz
                        'angle_deg'     : float(dev['angle']),         # 方位角 φ (°)
                        'beam_width_deg': float(dev['BeamWidth']),     # 半功率波束宽 (°)
                    }
                else:  # RECEIVER
                    rxs[key] = {
                        'id'                    : dev['ID'],
                        'position'              : pos,
                        'gain_dbi'              : float(dev['Gain']),
                        'freq_ghz'              : float(dev['Central_F']),
                        'bw_mhz'                : float(dev['Bandwith']),
                        # JSON 存储为正整数 (e.g. "100.0")，实际为 -100 dBm
                        'sensitivity_dbm'       : -float(dev['Sensitivity']),
                        'noise_figure_db'       : float(dev['noiseFigure']),
                        'interference_margin_db': float(dev['interferenceMargin']),
                    }
            nodes[usv_id] = {
                'id'          : usv_id,
                'location'    : [float(v) for v in loc],
                'transmitters': txs,
                'receivers'   : rxs,
            }
        return nodes

    def get_thermal_noise_floor(self, bw_mhz=None, nf_db=None):
        """
        热噪声基底 N_floor = kTB + NF (dBm)
        默认 BW=100 MHz, NF=3 dB → 约 -91 dBm
        """
        bw_hz = (bw_mhz or self.bw_mhz) * 1e6
        if nf_db is None:
            rx0 = next(iter(next(iter(self.nodes.values()))['receivers'].values()))
            nf_db = rx0['noise_figure_db']
        n_therm_dbm = 10.0 * np.log10(1.38e-23 * 290.0 * bw_hz) + 30.0  # W → dBm
        return n_therm_dbm + nf_db

    def make_scene_for_link(self, tx_pos, rx_pos, wind_speed=None):
        """
        为 TX-RX 对创建 SceneGenerator。
        lx = XY 平面水平距离；tx_height / rx_height 取绝对 z 坐标。
        Returns (scene, horiz_dist_m)
        """
        ws   = wind_speed if wind_speed is not None else self.wind_speed
        dist = float(np.sqrt((rx_pos[0]-tx_pos[0])**2 + (rx_pos[1]-tx_pos[1])**2))
        dist = max(dist, 1.0)
        scene = SceneGenerator(
            freq_ghz  = self.freq_ghz,
            lx        = dist,
            lz        = self.lz,
            dpml      = self.dpml,
            wind_speed= ws,
            fetch_km  = self.fetch_km,
            tx_height = float(tx_pos[2]),
            rx_height = float(rx_pos[2]),
        )
        return scene, dist

    def beam_width_to_sigma(self, tx_dev):
        """
        BeamWidth (°) → PE 高斯源宽度 σ_z (m)
        σ = λ / (2π · sin(θ_half))，裁剪至 [0.05, 5.0] m
        """
        c    = 299792458.0
        lam  = c / (tx_dev['freq_ghz'] * 1e9)
        half = np.radians(tx_dev['beam_width_deg'] / 2.0)
        return float(np.clip(lam / (2.0 * np.pi * max(np.sin(half), 0.01)), 0.05, 5.0))

    def beam_gain_db(self, tx_dev, rx_pos):
        """
        方位波束增益修正 (dB)
        主瓣内 0 dB；超出半波束宽后高斯旁瓣：G = -12·(Δφ/θ_half)²，下限 -25 dB
        """
        tp   = tx_dev['position']
        az   = np.degrees(np.arctan2(float(rx_pos[1]-tp[1]),
                                      float(rx_pos[0]-tp[0]))) % 360.0
        ctr  = tx_dev['angle_deg'] % 360.0
        hbw  = tx_dev['beam_width_deg'] / 2.0
        diff = abs(az - ctr) % 360.0
        if diff > 180.0:
            diff = 360.0 - diff
        if diff <= hbw:
            return 0.0
        return float(max(-12.0 * ((diff - hbw) / max(hbw, 1e-6))**2, -25.0))

    @property
    def all_tx(self):
        return [(uid, tk, td) for uid, nd in self.nodes.items()
                for tk, td in nd['transmitters'].items()]

    @property
    def all_rx(self):
        return [(uid, rk, rd) for uid, nd in self.nodes.items()
                for rk, rd in nd['receivers'].items()]


In [43]:
# ==========================================
# 7. EMC 指标计算器 (EMCMetricsComputer)
#    四大指标：SCF / S3I / T_elevation / D_desense
# ==========================================
class EMCMetricsComputer:
    """
    所有方法均为静态方法。
    接收功率公式 (2D 柱面波 Friis + PE 多径修正):
        P_rx = P_tx + G_tx + G_rx + G_beam
             + 20·log10(λ/4π) - 10·log10(r)    [柱面波自由空间基准]
             + 20·log10(|u_PE(r)|)              [PE 多径传播修正]
    |u_PE(r)| 为 PE 归一化减少场 (已补偿 1/√r 几何扩展)。
    """

    # ── 公共 PE 接口 ─────────────────────────────────────────────────────────
    @staticmethod
    def run_pe_link(scene, tx_h, rx_h, beam_sigma_m=0.275, dx=1.0):
        """
        为单条链路运行 PE 求解器。
        dx=1.0 m 步进 (EMC 精度)。
        Returns (pe_range, pe_1d_mag, e_at_end)
        """
        pe_solver = PESolver(scene, dx=dx, dz=0.1)
        pe_x = scene.x_full - scene.x_full[0]
        pe_h = scene.h_full.copy()
        pe_solver.init_gaussian_source(tx_h, pe_h[0], beam_width=beam_sigma_m)
        pe_range, pe_1d_mag, _, _, _ = pe_solver.march(pe_x, pe_h, scene.lx, rx_h)
        idx_end = int(np.argmin(np.abs(pe_range - scene.lx)))
        return pe_range, pe_1d_mag, float(pe_1d_mag[idx_end])

    @staticmethod
    def friis_pe_power_dbm(e_mag, range_m, p_tx_dbm, g_tx_dbi,
                            g_rx_dbi, g_beam_db, freq_ghz, isolation_db=0.0):
        """
        Friis 链路方程 + PE 传播因子 → 接收功率 (dBm)
        P_rx = P_tx + G_tx + G_rx + G_beam + FSPL_cyl + PE_corr - isolation_db
        isolation_db: 额外隔离度（dB），如滤波器、屏蔽、极化隔离等
        """
        lam      = 299792458.0 / (freq_ghz * 1e9)
        r        = max(range_m, 1.0)
        fspl_cyl = 20.0 * np.log10(lam / (4.0 * np.pi)) - 10.0 * np.log10(r)
        pe_corr  = 20.0 * np.log10(float(e_mag) + 1e-20)
        return p_tx_dbm + g_tx_dbi + g_rx_dbi + g_beam_db + fspl_cyl + pe_corr + isolation_db

    # ════════════════════════════════════════════════════════════════════════
    # 指标 1 — 系统级电磁耦合度 SCF
    # SCF = (1/M×N) Σ_{i,j,j≠i} (P_rx^{i,j} - N_floor)
    # ════════════════════════════════════════════════════════════════════════
    @staticmethod
    def compute_scf(fsg, verbose=True):
        """
        遍历所有跨 USV TX→RX 对，计算耦合功率矩阵，
        返回超出热噪声基底的平均耦合功率 SCF (dB)。

        Returns
        -------
        scf_db        : float  SCF 标量值
        coupling_dict : dict   {(tx_usv_id, rx_usv_id): P_rx_dBm}
        n_floor_dbm   : float  热噪声基底 (dBm)
        """
        n_floor = fsg.get_thermal_noise_floor()
        coupling_dict = {}
        above_list    = []

        if verbose:
            print(f"  N_floor = {n_floor:.2f} dBm")

        for rx_uid, rx_nd in fsg.nodes.items():
            for rx_key, rx_dev in rx_nd['receivers'].items():
                rx_pos = rx_dev['position']

                for tx_uid, tx_nd in fsg.nodes.items():
                    if tx_uid == rx_uid:
                        continue                     # 排除自身平台链路
                    for tx_key, tx_dev in tx_nd['transmitters'].items():
                        tp   = tx_dev['position']
                        dist = float(np.sqrt((rx_pos[0]-tp[0])**2
                                             + (rx_pos[1]-tp[1])**2))
                        if dist < 1.0:
                            continue

                        sigma_m = fsg.beam_width_to_sigma(tx_dev)
                        g_beam  = fsg.beam_gain_db(tx_dev, rx_pos)

                        scene, _ = fsg.make_scene_for_link(tp, rx_pos)
                        _, _, e_rx = EMCMetricsComputer.run_pe_link(
                            scene, float(tp[2]), float(rx_pos[2]), sigma_m)

                        p_rx = EMCMetricsComputer.friis_pe_power_dbm(
                            e_rx, dist,
                            tx_dev['power_dbm'], tx_dev['gain_dbi'],
                            rx_dev['gain_dbi'], g_beam, tx_dev['freq_ghz'],
                            rx_dev['interference_margin_db'])

                        coupling_dict[(tx_uid, rx_uid)] = p_rx
                        above_list.append(p_rx - n_floor)

                        if verbose:
                            print(f"    {tx_uid}→{rx_uid}  "
                                  f"d={dist:6.0f}m  P_rx={p_rx:6.1f} dBm  "
                                  f"超底噪={p_rx-n_floor:+5.1f} dB  "
                                  f"波束修正={g_beam:+5.1f} dB")

        scf_db = float(np.mean(above_list)) if above_list else 0.0
        print(f"\n  ✅ SCF = {scf_db:.2f} dB  ({len(above_list)} 条跨节点干扰链路)")
        return scf_db, coupling_dict, n_floor

    # ════════════════════════════════════════════════════════════════════════
    # 指标 2 — 海况敏感度指数 S3I
    # S3I = (1/K) Σ_k |P_rough(x_k) - P_flat(x_k)|
    # ════════════════════════════════════════════════════════════════════════
    @staticmethod
    def compute_s3i(fsg, wind_speed_rough=3.0,
                    ref_tx_usv='USV1', ref_rx_usv='USV2', verbose=True):
        """
        对参考链路 (默认 USV1→USV2) 对比平静海面 (WS=0.5 m/s) 与
        指定海况 (wind_speed_rough) 下的场强，计算空间平均差异 S3I (dB)。

        Returns
        -------
        s3i_db      : float
        pe_flat_db  : ndarray  平静海面场强剖面 (dB)
        pe_rough_db : ndarray  粗糙海面场强剖面 (dB)
        pe_range    : ndarray  距离轴 (m)
        """
        tx_dev = next(iter(fsg.nodes[ref_tx_usv]['transmitters'].values()))
        rx_dev = next(iter(fsg.nodes[ref_rx_usv]['receivers'].values()))
        tp, rp = tx_dev['position'], rx_dev['position']
        sigma_m = fsg.beam_width_to_sigma(tx_dev)
        dist    = float(np.sqrt((rp[0]-tp[0])**2 + (rp[1]-tp[1])**2))

        # 平静海面
        sc_flat, _ = fsg.make_scene_for_link(tp, rp, wind_speed=0.5)
        rng_f, mag_f, _ = EMCMetricsComputer.run_pe_link(
            sc_flat, float(tp[2]), float(rp[2]), sigma_m)

        # 粗糙海面
        sc_rough, _ = fsg.make_scene_for_link(tp, rp, wind_speed=wind_speed_rough)
        rng_r, mag_r, _ = EMCMetricsComputer.run_pe_link(
            sc_rough, float(tp[2]), float(rp[2]), sigma_m)

        # 对齐到相同距离轴
        n           = min(len(mag_f), len(mag_r))
        pe_range    = rng_f[:n]
        pe_flat_db  = 20.0 * np.log10(mag_f[:n] + 1e-20)
        pe_rough_db = 20.0 * np.log10(mag_r[:n] + 1e-20)

        valid  = (pe_range > 10.0) & np.isfinite(pe_flat_db) & np.isfinite(pe_rough_db)
        s3i_db = float(np.mean(np.abs(pe_rough_db[valid] - pe_flat_db[valid])))

        if verbose:
            print(f"  ✅ S3I = {s3i_db:.3f} dB  "
                  f"(链路 {ref_tx_usv}→{ref_rx_usv}, d={dist:.0f} m, "
                  f"WS 0.5→{wind_speed_rough} m/s)")
        return s3i_db, pe_flat_db, pe_rough_db, pe_range

    # ════════════════════════════════════════════════════════════════════════
    # 指标 3 — 背景噪声抬升热图 T_elevation
    # T_elev(x,y) = 10·log10[(I_total + N_therm) / N_therm]
    # ════════════════════════════════════════════════════════════════════════
    @staticmethod
    def compute_noise_elevation_map(fsg, grid_resolution=50, verbose=True):
        """
        在编队覆盖海域网格上叠加所有 TX 的非相干干扰功率，
        计算相对热噪声的背景噪声抬升量。

        策略：为每个 TX 预计算 PE 1D 传播曲线 (各向同性接收 G_rx=0 dBi 背景场)，
        再矢量化查表求各网格点的总干扰功率。

        Returns
        -------
        X_grid, Y_grid : ndarray  网格坐标 (m)
        T_elev         : ndarray  噪声抬升量 (dB)
        I_total_mw     : ndarray  总干扰功率密度 (mW)，供 D_desense 复用
        """
        # ── 覆盖网格 ─────────────────────────────────────────────────────
        all_loc = [nd['location'] for nd in fsg.nodes.values()]
        margin  = 300
        x0 = min(p[0] for p in all_loc) - margin
        x1 = max(p[0] for p in all_loc) + margin
        y0 = min(p[1] for p in all_loc) - margin
        y1 = max(p[1] for p in all_loc) + margin
        xs = np.arange(x0, x1 + grid_resolution, float(grid_resolution))
        ys = np.arange(y0, y1 + grid_resolution, float(grid_resolution))
        X_grid, Y_grid = np.meshgrid(xs, ys)

        n_therm_mw = 10.0 ** (fsg.get_thermal_noise_floor() / 10.0)

        # ── 为每个 TX 预计算 PE 1D 传播曲线 ─────────────────────────────
        curves = {}
        for uid, tk, td in fsg.all_tx:
            tp = td['position']
            corners = [(x0,y0),(x0,y1),(x1,y0),(x1,y1)]
            max_r = float(np.clip(
                max(np.sqrt((cx-tp[0])**2 + (cy-tp[1])**2) for cx,cy in corners),
                100.0, 4000.0))
            sigma_m = fsg.beam_width_to_sigma(td)
            # 背景场：接收高度取典型海面观测高 2 m
            sc = SceneGenerator(
                freq_ghz  = td['freq_ghz'],
                lx        = max_r,
                lz        = fsg.lz,
                dpml      = fsg.dpml,
                wind_speed= fsg.wind_speed,
                fetch_km  = fsg.fetch_km,
                tx_height = float(tp[2]),
                rx_height = 2.0,
            )
            pe_rng, pe_mag, _ = EMCMetricsComputer.run_pe_link(
                sc, float(tp[2]), 2.0, sigma_m)
            curves[(uid, tk)] = (pe_rng, pe_mag, td)
            if verbose:
                print(f"  📡 TX {uid}/{tk}  PE曲线完毕 (max_r={max_r:.0f} m)")

        # ── 矢量化叠加各 TX 干扰 ─────────────────────────────────────────
        I_mw = np.zeros_like(X_grid, dtype=float)
        for (uid, tk), (pe_rng, pe_mag, td) in curves.items():
            tp  = td['position']
            lam = 299792458.0 / (td['freq_ghz'] * 1e9)

            R = np.maximum(
                np.sqrt((X_grid - tp[0])**2 + (Y_grid - tp[1])**2), 1.0)

            E = np.interp(R.ravel(), pe_rng, pe_mag,
                          left=float(pe_mag[0]),
                          right=float(pe_mag[-1])).reshape(R.shape)

            # 方位波束增益修正（矢量化）
            AZ   = np.degrees(np.arctan2(Y_grid - tp[1], X_grid - tp[0])) % 360.0
            ctr  = td['angle_deg'] % 360.0
            hbw  = td['beam_width_deg'] / 2.0
            diff = np.abs(AZ - ctr) % 360.0
            diff = np.where(diff > 180.0, 360.0 - diff, diff)
            exc  = np.maximum((diff - hbw) / max(hbw, 1e-6), 0.0)
            Gbm  = np.where(diff <= hbw, 0.0, np.maximum(-12.0 * exc**2, -25.0))

            # 柱面波路径损耗 + PE 修正 (G_rx=0 dBi 背景场)
            path  = 20.0 * np.log10(lam / (4.0 * np.pi)) - 10.0 * np.log10(R)
            pe_c  = 20.0 * np.log10(E + 1e-20)
            p_dbm = td['power_dbm'] + td['gain_dbi'] + 0.0 + Gbm + path + pe_c
            I_mw += 10.0 ** (p_dbm / 10.0)

        T_elev = 10.0 * np.log10((I_mw + n_therm_mw) / n_therm_mw)
        if verbose:
            print(f"\n  ✅ T_elevation: min={T_elev.min():.1f} dB, "
                  f"max={T_elev.max():.1f} dB, "
                  f"mean={T_elev.mean():.1f} dB")
        return X_grid, Y_grid, T_elev, I_mw

    # ════════════════════════════════════════════════════════════════════════
    # 指标 4 — 接收机灵敏度恶化分布图 D_desense
    # D = max(I_rx(x,y) - S_sens, 0)
    # ════════════════════════════════════════════════════════════════════════
    @staticmethod
# 在 EMCMetricsComputer 类中替换原有方法
    def compute_desense_map(X_grid, Y_grid, I_total_mw, victim_rx_dev):
        s_sens   = float(victim_rx_dev['sensitivity_dbm'])
        g_rx     = float(victim_rx_dev['gain_dbi'])
        
        # 计算接收机端口干扰电平 (dBm)
        I_rx_dbm = 10.0 * np.log10(np.maximum(I_total_mw, 1e-30)) + g_rx
        
        # 计算恶化量 D (dB)
        D_des = np.maximum(I_rx_dbm - s_sens, 0.0)

        # --- 新增：单位面积评价指标 ---
        # 计算网格面积 (m2)
        dx = X_grid[0, 1] - X_grid[0, 0]
        dy = Y_grid[1, 0] - Y_grid[0, 0]
        total_area = (X_grid.max() - X_grid.min()) * (Y_grid.max() - Y_grid.min())
        
        # 单位面积平均恶化强度 (dB/m2)
        adi = np.sum(D_des * dx * dy) / total_area
        
        # 覆盖率统计
        n_exceed = int(np.sum(D_des > 0.1))
        pct = 100.0 * n_exceed / D_des.size
        
        print(f"  ✅ D_desense 评估完毕:")
        print(f"    - 受害阈值 S_sens: {s_sens:.0f} dBm")
        print(f"    - 危险区域占比: {pct:.1f}%")
        print(f"    - 单位面积平均恶化强度 (ADI): {adi:.4f} dB/m²") # 核心修正点
        print(f"    - 峰值恶化量: {D_des.max():.2f} dB")
        
        return D_des, adi

In [44]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects  # 显式导入 patheffects
import numpy as np
from matplotlib import rcParams

# ==========================================
# 全局科研绘图风格配置
# ==========================================
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "SimSun"], # 优先Times, 中文宋体
    "mathtext.fontset": "stix",       # 使 LaTeX 字体与 Times 风格统一
    "axes.linewidth": 1.0,            # 边框粗细
    "xtick.direction": "in",          # 刻度线向内
    "ytick.direction": "in",
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "savefig.dpi": 300,               # 默认导出 300 DPI
    "figure.autolayout": False        # 手动控制 layout 避免标题截断
})

class EMCVisualizer:

    # ── Fig_EMC1: 系统耦合矩阵热图 (SCF) ─────────────────────────────────────
    @staticmethod
    def plot_coupling_matrix(coupling_dict, n_floor_dbm, usv_ids, scf_db, case_tag=''):
        """
        优化点：使用感知均匀色图 'magma'，改进对比度文字显示，LaTeX 化物理指标
        """
        n = len(usv_ids)
        idx = {uid: i for i, uid in enumerate(usv_ids)}
        mat = np.full((n, n), np.nan)
        for (tx_uid, rx_uid), p_rx in coupling_dict.items():
            r, c = idx.get(rx_uid), idx.get(tx_uid)
            if r is not None and c is not None:
                mat[r, c] = p_rx - n_floor_dbm

        fig, ax = plt.subplots(figsize=(8, 6.5))
        vmin = np.nanmin(mat) if not np.all(np.isnan(mat)) else 0
        vmax = np.nanmax(mat) if not np.all(np.isnan(mat)) else 100
        
        # 使用 'viridis' 或 'magma' 具有更好的科研一致性
        im = ax.imshow(mat, cmap='magma', aspect='equal', vmin=vmin, vmax=vmax)
        
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(r'$P_{rx} - N_{floor}$ (dB)', fontweight='bold')

        ax.set_xticks(range(n)); ax.set_xticklabels(usv_ids, rotation=45, ha='right')
        ax.set_yticks(range(n)); ax.set_yticklabels(usv_ids)
        ax.set_xlabel('Transmitter (TX) USV')
        ax.set_ylabel('Receiver (RX) USV')
        
        title = (rf'$\bf{{SCF}}$ = {scf_db:.2f} dB above $N_{{floor}}$' + '\n' +
                 rf'($N_{{floor}} = {n_floor_dbm:.1f}$ dBm)')
        if case_tag:
            title = f'[{case_tag}] ' + title
        ax.set_title(title, pad=15)

        # 文字自适应背景色
        mid = (vmin + vmax) / 2.0
        for r in range(n):
            for c in range(n):
                if not np.isnan(mat[r, c]):
                    color = 'black' if mat[r, c] > (vmax - (vmax-vmin)*0.3) else 'white'
                    ax.text(c, r, f'{mat[r,c]:.1f}', ha='center', va='center',
                            fontsize=9, color=color, fontweight='bold')

        plt.tight_layout()
        fname = f'Fig_EMC1_SCF_CouplingMatrix_{case_tag}.png' if case_tag else 'Fig_EMC1_SCF_CouplingMatrix.png'
        plt.savefig(fname, bbox_inches='tight')
        plt.close()

    # ── Fig_EMC2: 海况敏感度 S3I 对比曲线 ─────────────────────────────────────
    @staticmethod
    def plot_s3i(pe_range, pe_flat_db, pe_rough_db, s3i_db,
                 ref_link=('USV1', 'USV2'), wind_rough=3.0, case_tag=''):
        """
        优化点：增强线宽，使用专业的颜色对（蓝色/红色），物理变量 LaTeX 化
        """
        fig, ax = plt.subplots(figsize=(10, 4.5))
        
        # 绘图曲线
        ax.plot(pe_range, pe_flat_db, color='#1f77b4', lw=1.5, label=r'Calm Sea ($W_s$ = 0.5 m/s)')
        ax.plot(pe_range, pe_rough_db, color='#d62728', lw=1.5, linestyle='--', 
                label=rf'Rough Sea ($W_s$ = {wind_rough} m/s)')
        
        # 填充差异
        ax.fill_between(pe_range, pe_flat_db, pe_rough_db, alpha=0.15, color='gray',
                        label=rf'Gap ($S^3I$ = {s3i_db:.3f} dB)')
        
        # 平均线
        ax.axhline(y=np.nanmean(pe_flat_db), color='#1f77b4', lw=1, linestyle=':', alpha=0.5)
        ax.axhline(y=np.nanmean(pe_rough_db), color='#d62728', lw=1, linestyle=':', alpha=0.5)

        title = rf'$\bf{{S^3I}}$ = {s3i_db:.3f} dB (Link: {ref_link[0]} $\rightarrow$ {ref_link[1]})'
        if case_tag:
            title = f'[{case_tag}] ' + title
        ax.set_title(title)
        ax.set_xlabel('Range $r$ (m)')
        ax.set_ylabel('Normalized Field Strength (dB)')
        
        ax.legend(loc='lower left', frameon=True, edgecolor='gray', fontsize=9)
        ax.grid(True, which='both', linestyle='--', alpha=0.3)
        ax.minorticks_on()
        
        plt.tight_layout()
        fname = f'Fig_EMC2_S3I_SeaStateSensitivity_{case_tag}.png' if case_tag else 'Fig_EMC2_S3I_SeaStateSensitivity.png'
        plt.savefig(fname)
        plt.close()

    # ── Fig_EMC3: 背景噪声抬升热图 T_elevation ────────────────────────────────
    @staticmethod
    def plot_noise_elevation_map(X_grid, Y_grid, T_elev, fsg, case_tag=''):
        """
        优化点：使用 'inferno' 模拟电磁热度，优化等值线标注，标准化节点符号
        """
        fig, ax = plt.subplots(figsize=(10, 7.5))
        
        # 等值线填充
        cf = ax.contourf(X_grid, Y_grid, T_elev, levels=40, cmap='inferno', extend='max')
        cbar = fig.colorbar(cf, ax=ax, fraction=0.03, pad=0.04)
        cbar.set_label(r'$T_{elevation}$ (dB)', fontweight='bold')

        # 核心等值线绘制
        cs = ax.contour(X_grid, Y_grid, T_elev, levels=[6, 12, 24, 36], 
                        colors='white', linewidths=0.8, alpha=0.5)
        ax.clabel(cs, fmt='%d dB', fontsize=9, inline=True)

        # USV 节点符号优化
        for uid, nd in fsg.nodes.items():
            loc = nd['location']
            ax.scatter(loc[0], loc[1], marker='s', s=80, facecolor='white', 
                       edgecolor='black', linewidth=1.5, zorder=10)
            ax.annotate(rf'$\mathbf{{{uid}}}$', (loc[0], loc[1]), xytext=(8, 8),
            textcoords='offset points', fontsize=10, color='white',
            path_effects=[path_effects.withStroke(linewidth=2, foreground='black')]) # 直接调用导入的变量

        ax.set_xlabel('$X$ (m)'); ax.set_ylabel('$Y$ (m)')
        title = 'Background Noise Elevation Map ($T_{elev}$)'
        if case_tag:
            title += f' — {case_tag}'
        ax.set_title(title, pad=10)
        ax.set_aspect('equal')
        
        plt.tight_layout()
        fname = f'Fig_EMC3_NoiseElevationMap_{case_tag}.png' if case_tag else 'Fig_EMC3_NoiseElevationMap.png'
        plt.savefig(fname, bbox_inches='tight')
        plt.close()

    # ── Fig_EMC4: 接收机灵敏度恶化分布图 D_desense ────────────────────────────
    @staticmethod
    def plot_desense_map(X_grid, Y_grid, D_desense, fsg, adi_val, victim_usv='USV1', victim_rx_key='Receiver1', case_tag=''):
        victim_dev = fsg.nodes[victim_usv]['receivers'][victim_rx_key]
        s_sens = victim_dev['sensitivity_dbm']

        fig, ax = plt.subplots(figsize=(11, 8))

        # 定义区域阈值
        levels = [0, 0.5, 5, 15, 30, 50, 80, 120]
        # 自定义色板：绿色(安全) -> 橙色(警告) -> 红色(危险)
        from matplotlib.colors import LinearSegmentedColormap
        colors = ["#e9f5e9", "#fff3e0", "#ffcc80", "#ffab91", "#f44336", "#b71c1c", "#4a148c"]
        custom_cmap = LinearSegmentedColormap.from_list("EMC_Risk", colors, N=len(levels)-1)

        # 绘制填充图
        cf = ax.contourf(X_grid, Y_grid, D_desense, levels=levels, cmap=custom_cmap, extend='max')
        
        # 绘制关键隔离等值线：判定“部署区域”的边界
        cs = ax.contour(X_grid, Y_grid, D_desense, levels=[0.5, 20], colors=['#2e7d32', '#c62828'], 
                        linewidths=[1.5, 2.0], linestyles=['--', '-'])
        ax.clabel(cs, fmt={0.5: 'Safe Boundary', 20: 'Danger Zone'}, fontsize=10, inline=True)

        cbar = fig.colorbar(cf, ax=ax, fraction=0.03, pad=0.04)
        cbar.set_label(r'$D_{desense}$ (dB)', fontweight='bold')

        # 标注节点与波束（逻辑同前）
        for uid, nd in fsg.nodes.items():
            loc = nd['location']
            is_vic = (uid == victim_usv)
            ax.scatter(loc[0], loc[1], marker='p' if is_vic else 'o', s=250 if is_vic else 100, 
                    facecolor='#FFD700' if is_vic else '#2c3e50', edgecolor='black', zorder=15)
            ax.annotate(uid, (loc[0], loc[1]), xytext=(10, 10), textcoords='offset points', fontweight='bold')

        ax.set_xlabel('$X$ (m)'); ax.set_ylabel('$Y$ (m)')
        title = rf'Receiver Desensitization Map ($D_{{desense}}$) — {case_tag}'
        sub_title = (rf'Target: {victim_usv} ($S_{{sens}} = {s_sens:.0f}$ dBm) | ' +
                    rf'$\bf{{ADI}} = {adi_val:.4f}$ dB/m$^2$')
        ax.set_title(title + '\n' + sub_title, pad=12)
        ax.set_aspect('equal')
        
        # 添加图例说明区域含义
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], color='#2e7d32', lw=2, linestyle='--', label='Deployable Region (D < 0.5dB)'),
            Line2D([0], [0], color='#c62828', lw=2, label='Hazardous Region (D > 20dB)')
        ]
        ax.legend(handles=legend_elements, loc='upper right', fontsize=9, framealpha=0.8)

        plt.tight_layout()
        plt.savefig(f'Fig_EMC4_DesensMap_Enhanced_{case_tag}.png', bbox_inches='tight')
        plt.close()

    # ── Fig_EMC5: 编队拓扑图 (辅助验证) ──────────────────────────────────────
    @staticmethod
    def plot_formation_topology(fsg, case_tag=''):
        """
        优化点：简洁的矢量风格，使用与编队尺度成正比的波束箭头，优化颜色循环
        """
        fig, ax = plt.subplots(figsize=(8, 8))
        
        # 使用科研标准的调色板
        prop_cycle = plt.rcParams['axes.prop_cycle']
        colors = prop_cycle.by_key()['color']

        # 计算编队尺度，用于动态调整箭头大小
        all_x = [nd['location'][0] for nd in fsg.nodes.values()]
        all_y = [nd['location'][1] for nd in fsg.nodes.values()]
        formation_scale = max(max(all_x) - min(all_x), max(all_y) - min(all_y))
        arrow_length = formation_scale * 0.07  # 箭头长度为编队尺度的 7%
        head_width = arrow_length * 0.2
        head_length = arrow_length * 0.3

        for i, (uid, nd) in enumerate(fsg.nodes.items()):
            loc = nd['location']
            col = colors[i % len(colors)]
            
            # 节点本体
            ax.scatter(loc[0], loc[1], s=150, color=col, edgecolor='white', linewidth=1.5, zorder=5)

            # TX 箭头：与编队尺度成正比
            for tk, td in nd['transmitters'].items():
                az_rad = np.radians(td['angle_deg'])
                dx, dy = arrow_length * np.cos(az_rad), arrow_length * np.sin(az_rad)
                ax.arrow(loc[0], loc[1], dx, dy, head_width=head_width, head_length=head_length, 
                         fc=col, ec=col, alpha=0.6, length_includes_head=True)
                
                # TX 物理位置
                tp = td['position']
                ax.scatter(tp[0], tp[1], marker='^', s=40, color=col, zorder=6)

            # RX 物理位置
            for rk, rd in nd['receivers'].items():
                rp = rd['position']
                ax.scatter(rp[0], rp[1], marker='v', s=40, color=col, zorder=6)

            ax.annotate(uid, (loc[0], loc[1]), xytext=(8, 8), textcoords='offset points',
                        fontsize=11, fontweight='bold', color=col)

        ax.set_xlabel('$X$ (m)'); ax.set_ylabel('$Y$ (m)')
        title = r'Formation Topology ($\bf{\Delta}$: TX, $\bf{\nabla}$: RX, $\bf{\rightarrow}$: Beam)'
        if case_tag:
            title += f' — {case_tag}'
        ax.set_title(title, pad=15)
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.set_aspect('equal')
        
        plt.tight_layout()
        fname = f'Fig_EMC0_FormationTopology_{case_tag}.png' if case_tag else 'Fig_EMC0_FormationTopology.png'
        plt.savefig(fname, bbox_inches='tight')
        plt.close()

In [45]:
# ==========================================
# 9. 编队 EMC 分析主程序
#    支持 A/B 双 JSON 文件对比计算
# ==========================================
if __name__ == "__main__":
    JSONA_PATH   = 'Test_A.jsonc'   # Case A: 违反 GJB 的高危态
    JSONB_PATH   = 'Test_B.jsonc'   # Case B: 符合 GJB 的安全态
    WIND_SPEED   = 3.0   # 当前仿真海况风速 (m/s)，用于 SCF / T_elev / D_desense
    WIND_ROUGH   = 3.0   # S3I 对照粗糙海况风速 (m/s)
    VICTIM_USV   = 'USV1'
    VICTIM_RX    = 'Receiver1'   # Device2

    cases = [
        (JSONA_PATH, 'CaseA'),
        (JSONB_PATH, 'CaseB'),
    ]

    for json_path, case_tag in cases:
        SEP = '=' * 65
        print(f'\n{SEP}')
        print(f'  编队 EMC 全链路分析  |  {case_tag}  |  PE-PLST 传播模型  |  7 USV 编队')
        print(SEP)

        # ── 加载编队场景 ─────────────────────────────────────────────────────────
        fsg     = FormationSceneGenerator(json_path, wind_speed=WIND_SPEED)
        usv_ids = list(fsg.nodes.keys())

        # ── 辅助拓扑图 (验证 JSON 解析) ──────────────────────────────────────────
        print('\n绘制编队拓扑图...')
        EMCVisualizer.plot_formation_topology(fsg, case_tag=case_tag)

        # ── 指标 1: SCF ──────────────────────────────────────────────────────────
        print(f'\n{"─"*65}')
        print(f'[{case_tag}] [1/4] 系统级电磁耦合度 (System Coupling Factor, SCF) ...')
        scf_db, coupling_dict, n_floor = EMCMetricsComputer.compute_scf(fsg, verbose=True)

        # ── 指标 2: S3I ──────────────────────────────────────────────────────────
        print(f'\n{"─"*65}')
        print(f'[{case_tag}] [2/4] 海况敏感度指数 (Sea-State Sensitivity Index, S3I) ...')
        s3i_db, pe_flat, pe_rough, rng_s3i = EMCMetricsComputer.compute_s3i(
            fsg,
            wind_speed_rough = WIND_ROUGH,
            ref_tx_usv = 'USV1',
            ref_rx_usv = 'USV2',
        )

        # ── 指标 3: T_elevation ──────────────────────────────────────────────────
        print(f'\n{"─"*65}')
        print(f'[{case_tag}] [3/4] 背景噪声抬升热图 (Background Noise Elevation Map) ...')
        X_g, Y_g, T_elev, I_mw = EMCMetricsComputer.compute_noise_elevation_map(
            fsg, grid_resolution=50, verbose=True)

# ── 指标 4: D_desense ────────────────────────────────────────────────────
        print(f'\n{"─"*65}')
        print(f'[{case_tag}] [4/4] 接收机灵敏度恶化分布图 (Receiver Desensitization Map) ...')
        print(f'      受害接收机: {VICTIM_USV}/{VICTIM_RX}')
        victim_rx_dev = fsg.nodes[VICTIM_USV]['receivers'][VICTIM_RX]
        
        # ✅ 修复 1：同时接收 D_des (数组) 和 adi_val (数值)
        D_des, adi_val = EMCMetricsComputer.compute_desense_map(X_g, Y_g, I_mw, victim_rx_dev)

        # ── 绘制四张 EMC 图表 ────────────────────────────────────────────────────
        print(f'\n{"─"*65}')
        print(f'[{case_tag}] 绘制四张 EMC 图表...')
        EMCVisualizer.plot_coupling_matrix(coupling_dict, n_floor, usv_ids, scf_db, case_tag=case_tag)
        EMCVisualizer.plot_s3i(rng_s3i, pe_flat, pe_rough, s3i_db,
                                ref_link=('USV1', 'USV2'), wind_rough=WIND_ROUGH, case_tag=case_tag)
        EMCVisualizer.plot_noise_elevation_map(X_g, Y_g, T_elev, fsg, case_tag=case_tag)
        
        # ✅ 修复 2：传入必填参数 adi_val
        EMCVisualizer.plot_desense_map(X_g, Y_g, D_des, fsg, adi_val,
                                        victim_usv=VICTIM_USV,
                                        victim_rx_key=VICTIM_RX,
                                        case_tag=case_tag)


        # ── 汇总报告 ─────────────────────────────────────────────────────────────
        print(f'\n{SEP}')
        print(f'  📊 编队 EMC 分析汇总报告 — {case_tag}')
        print(SEP)
        print(f'  配置文件                   = {json_path}')
        print(f'  海况风速 (WS)              = {WIND_SPEED} m/s')
        print(f'  受害接收机                 = {VICTIM_USV}/{VICTIM_RX}  '
              f'(S_sens = {victim_rx_dev["sensitivity_dbm"]:.0f} dBm)')
        print(f'  N_floor (热噪声基底)       = {n_floor:.2f} dBm')
        print(SEP)
        print(f'  SCF  (系统电磁耦合度)      = {scf_db:>8.2f} dB above N_floor')
        print(f'  S3I  (海况敏感度指数)      = {s3i_db:>8.3f} dB  '
              f'(WS 0.5→{WIND_ROUGH} m/s)')
        print(f'  T_elev (噪声抬升峰值)      = {T_elev.max():>8.1f} dB')
        print(f'  D_desense (灵敏度恶化峰值) = {D_des.max():>8.2f} dB')
        print(SEP)

        # EMC 评价建议
        print('\n  📋 EMC 评价建议:')
        if scf_db > 20:
            print('  ⚠️  SCF 较高，建议采用频谱分离或扩大编队安全距离减少同频干扰。')
        else:
            print('  ✅  SCF 处于可接受范围，编队内频谱规划合理。')
        if s3i_db > 3.0:
            print(f'  ⚠️  S3I={s3i_db:.2f} dB，海况变化对传播影响显著，'
                  '建议恶劣海况下增加通信余量。')
        else:
            print(f'  ✅  S3I={s3i_db:.2f} dB，编队电磁环境对海况不敏感。')
        if D_des.max() > 0:
            n_ex = int(np.sum(D_des > 0))
            print(f'  ⚠️  {VICTIM_USV}/{VICTIM_RX} 存在 {n_ex} 个超阈值网格点，'
                  '建议加强易扰设备的屏蔽隔离。')
        else:
            print(f'  ✅  {VICTIM_USV}/{VICTIM_RX} 无灵敏度恶化，设备工作正常。')
        print(SEP)

    print('\n✅ 全部完成，所有 Case 的图表已保存。')



  编队 EMC 全链路分析  |  CaseA  |  PE-PLST 传播模型  |  7 USV 编队
✅ 编队场景加载完毕: 7 个 USV | f=1.0 GHz | BW=100.0 MHz | WS=3.0 m/s

绘制编队拓扑图...

─────────────────────────────────────────────────────────────────
[CaseA] [1/4] 系统级电磁耦合度 (System Coupling Factor, SCF) ...
  N_floor = -90.98 dBm
✅ 场景生成完毕: JONSWAP海面 (风速=3.0m/s)
    USV2→USV1  d=   399m  P_rx=  -7.6 dBm  超底噪=+83.4 dB  波束修正= +0.0 dB
✅ 场景生成完毕: JONSWAP海面 (风速=3.0m/s)
    USV3→USV1  d=   401m  P_rx=  -8.2 dBm  超底噪=+82.8 dB  波束修正= +0.0 dB
✅ 场景生成完毕: JONSWAP海面 (风速=3.0m/s)
    USV4→USV1  d=   799m  P_rx= -44.8 dBm  超底噪=+46.2 dB  波束修正=-25.0 dB
✅ 场景生成完毕: JONSWAP海面 (风速=3.0m/s)
    USV5→USV1  d=   801m  P_rx= -44.4 dBm  超底噪=+46.6 dB  波束修正=-25.0 dB
✅ 场景生成完毕: JONSWAP海面 (风速=3.0m/s)
    USV6→USV1  d=  1199m  P_rx= -52.4 dBm  超底噪=+38.6 dB  波束修正=-25.0 dB
✅ 场景生成完毕: JONSWAP海面 (风速=3.0m/s)
    USV7→USV1  d=  1201m  P_rx= -52.5 dBm  超底噪=+38.5 dB  波束修正=-25.0 dB
✅ 场景生成完毕: JONSWAP海面 (风速=3.0m/s)
    USV1→USV2  d=   401m  P_rx= -33.2 dBm  超底噪=+57.8 dB  波束修正=-25.0 dB
✅ 场景